In [ ]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region 
    for region, electrodes in REGION_TO_ELECTRODES.items() 
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

In [ ]:
import json
from pathlib import Path

BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subjects = ["sub-p0002"]
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4


HRF_DELAY = 6.0
WINDOW = 1

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"TR: {tr} s")

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_LOG = RESULTS_DIR / "4_Classes_ANN_analysis_log.txt"
log_file = open(OUTPUT_LOG, 'w', encoding='utf-8')
experiments_results = []

In [ ]:
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
import pandas as pd

all_events = []
subject = subjects[0]
for run in range(1, n_runs_per_subject + 1):
    events_path = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()

#eletrode -> region mapping
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

In [ ]:
region_counts = stim_events['region'].value_counts()
for region in ['Middle_Finger', 'Hand', 'Forearm', 'Arm']:
    count = region_counts.get(region, 0)
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {count} samples")

In [ ]:
from nilearn.image import load_img, index_img

first_run_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)
print(f"\nReference image shape: {first_run_img.shape}")

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import new_img_like, resample_to_img
from nilearn.maskers import NiftiMasker
from nilearn.plotting import plot_roi

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)
display = plot_roi(s1_mask_resampled, bg_img=ref_img, 
                            title='S1 ROI Mask (Left G_postcentral)',
                            display_mode='ortho', cut_coords=(0, -20, 60))
display.savefig(FIGURES_DIR / 's1_mask.png', dpi=150)
display.close()

In [ ]:
from nilearn.image import concat_imgs

volumes = []
y_list = []
run_labels = []

for run in range(1, n_runs_per_subject + 1):
    bold_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
    bold_img = load_img(str(bold_path))
    print(f"Run {run}...")
    run_events = stim_events[stim_events['run'] == run]
    for _, event in run_events.iterrows():
        onset = event["onset"]
        region = event["region"]
        target_time = onset + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))
        
        vol_indices = [vol_indx - WINDOW, vol_indx, vol_indx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vol_indices):
            imgs = [index_img(bold_img, v) for v in vol_indices]
            avg_data = np.mean([img.get_fdata() for img in imgs], axis=0)
            avg_img = new_img_like(imgs[0], avg_data)
            volumes.append(avg_img)
            y_list.append(REGION_TO_LABEL[region])
            run_labels.append(run)

In [ ]:
fmri_4d = concat_imgs(volumes)
y = np.array(y_list)
groups = np.array(run_labels)

print(f"\n4D fMRI shape: {fmri_4d.shape}")
print(f"Labels: {np.unique(y, return_counts=True)}")
print(f"Groups: {np.unique(groups, return_counts=True)}")

### Run the 4 binary searchlights, saves Nifti map

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from nilearn.image import mean_img

cv = LeaveOneGroupOut()
mean_fmri = mean_img(fmri_4d)

In [ ]:
OVA_DIR = RESULTS_DIR / "ova_maps"
OVA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from sklearn.svm import LinearSVC
from nilearn.decoding import SearchLight
ova_results = {}
class_names = ['Middle_Finger', 'Hand', 'Forearm', 'Arm']

for class_name in class_names:
    c = REGION_TO_LABEL[class_name]
    y_binary = (y == c).astype(int)
    n_pos, n_neg = y_binary.sum(), (1 - y_binary).sum()

    estimator = LinearSVC(C=3, class_weight='balanced', max_iter=10000)
    search_light = SearchLight(
        mask_img=s1_mask_resampled,
        process_mask_img=s1_mask_resampled,
        radius=4,
        estimator=estimator,
        cv=cv,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )
    search_light.fit(fmri_4d, y_binary, groups=groups)

    out_path = OVA_DIR / f"ova_scores_{class_name}.nii.gz"
    search_light.scores_img_.to_filename(str(out_path))
    ova_results[class_name] = search_light.scores_img_
    mask_bool = s1_mask_resampled.get_fdata().astype(bool)
    scores = search_light.scores_img_.get_fdata()[mask_bool]

    print(f"  Mean: {scores.mean():.4f}, Max: {scores.max():.4f}")
    print(f"  Voxels > 0.50: {(scores > 0.50).sum()} / {len(scores)}")
    print(f"  Voxels > 0.60: {(scores > 0.60).sum()} / {len(scores)}")

In [ ]:
# MNI coordinates from S1 somatotopy literature (left hemisphere)
# Martuzzi et al. (2014) NeuroImage; Blankenburg et al. (2003); Huang & Sereno (2018)
LITERATURE_MNI = {
    'Middle_Finger': (-44, -24, 50),
    'Hand':          (-40, -28, 56),
    'Forearm':       (-34, -32, 62),
    'Arm':           (-28, -34, 68),
}
lit_colors = {'Middle_Finger': 'cyan', 'Hand': 'yellow', 'Forearm': 'lime', 'Arm': 'white'}

In [ ]:
from nilearn.plotting import plot_stat_map
import matplotlib.pyplot as plt

for class_name in class_names:
    scores_img = ova_results[class_name]
    lit_coord = LITERATURE_MNI[class_name]

    fig, ax = plt.subplots(1, 1, figsize=(16, 6))
    disp = plot_stat_map(
        scores_img,
        bg_img=mean_fmri,
        title=f"OVA: {class_name} vs Rest  |  Literature coord: {lit_coord}",
        display_mode="ortho",
        cut_coords=lit_coord,  # center view on literature coordinate
        vmax=0.85,
        cmap="inferno",
        threshold=0.50,
        colorbar=True,
        black_bg=True,
        axes=ax,
    )

    # Mark literature coordinate
    disp.add_markers(
        marker_coords=[lit_coord],
        marker_color=lit_colors[class_name],
        marker_size=120,
    )

    fig.savefig(FIGURES_DIR / f'ova_{class_name}_with_lit.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
from nilearn.plotting import view_img
from IPython.display import display as ipy_display
from nibabel.affines import apply_affine

for class_name in class_names:
    scores_img = ova_results[class_name]
    lit_coord = LITERATURE_MNI[class_name]

    #create a marker for the literature coordinates 
    scores_data = scores_img.get_fdata().copy()
    inv_affine = np.linalg.inv(scores_img.affine)
    vi, vj, vk = np.round(apply_affine(inv_affine, lit_coord)).astype(int)

    #small sphere, radius 2 voxels with value 1.0
    for di in range(-2, 3):
        for dj in range(-2, 3):
            for dk in range(-2, 3):
                if di**2 + dj**2 + dk**2 <= 4:
                    ni, nj, nk = vi+di, vj+dj, vk+dk
                    if (0 <= ni < scores_data.shape[0] and
                        0 <= nj < scores_data.shape[1] and
                        0 <= nk < scores_data.shape[2]):
                        scores_data[ni, nj, nk] = 1.0

    marked_img = new_img_like(scores_img, scores_data)

    interactive = view_img(
        marked_img,
        bg_img=mean_fmri,
        title=f"OVA: {class_name} vs Rest",
        threshold=0.50,
        cmap="inferno",
        symmetric_cmap=False,
        vmax=1.0,
        cut_coords=lit_coord,
        colorbar=True,
        black_bg=True,
    )
    interactive.save_as_html(str(FIGURES_DIR / f'ova_{class_name}_interactive.html'))
    ipy_display(interactive)

### Winner Map
Combination of the 4 maps into one, each voxel is assigned to the class with the highest binary accuracy

In [ ]:
#4 OVA score maps into (X, Y, Z, 4)
mask_bool = s1_mask_resampled.get_fdata().astype(bool)
score_stack = np.stack(
    [ova_results[cn].get_fdata() for cn in class_names], axis=-1
)

In [ ]:
# Assign class with highest OVA accuracy
winner_labels = np.argmax(score_stack, axis=-1).astype(float)
max_accuracy = np.max(score_stack, axis=-1)

# Mark voxels below 0.50 as unclassified (NaN)
below_chance = max_accuracy <= 0.50
winner_labels[below_chance] = np.nan
winner_labels[~mask_bool] = np.nan

# max-accuracy map, only above-chance voxels
max_accuracy_masked = max_accuracy.copy()
max_accuracy_masked[below_chance] = np.nan
max_accuracy_masked[~mask_bool] = np.nan

combined_img = new_img_like(ref_img, winner_labels)
max_acc_img = new_img_like(ref_img, max_accuracy_masked)
combined_img.to_filename(str(OVA_DIR / 'winner_take_all_map.nii.gz'))
max_acc_img.to_filename(str(OVA_DIR / 'max_accuracy_map.nii.gz'))

In [ ]:
classified = winner_labels[mask_bool]
classified_valid = classified[~np.isnan(classified)]
print(f"Classified voxels: {len(classified_valid)} / {mask_bool.sum()}")
for i, cn in enumerate(class_names):
    n = (classified_valid == i).sum()
    print(f"  {cn}: {n} voxels ({100*n/len(classified_valid):.1f}%)")
print(f"Unclassified (all below 0.50): {mask_bool.sum() - len(classified_valid)}")

In [ ]:
from nilearn.plotting import plot_anat
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

colors = ['red', 'blue', 'green', 'purple']

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
display = plot_anat(
    mean_fmri,
    title="Winner-Take-All Somatotopic Map",
    display_mode="ortho",
    cut_coords=(0, -20, 60),
    black_bg=True,
    axes=ax,
)

for i, (class_name, color) in enumerate(zip(class_names, colors)):
    class_data = np.where(winner_labels == i, 1.0, np.nan)
    class_img = new_img_like(ref_img, class_data)
    display.add_overlay(class_img, cmap=ListedColormap([color]), alpha=0.7)

legend_patches = [mpatches.Patch(color=c, label=cn) for c, cn in zip(colors, class_names)]
fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=11)
fig.savefig(FIGURES_DIR / 'winner_take_all_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
from nilearn.plotting import view_img_on_surf

#labels 1-4, threshold at 0.5 (threshold cannot be negative), 0 class cannot be hidden
winner_shifted = winner_labels + 1  
combined_shifted_img = new_img_like(ref_img, winner_shifted)

colors = ['red', 'blue', 'green', 'purple']
cmap_discrete = ListedColormap(colors)

view = view_img_on_surf(
    combined_shifted_img,
    surf_mesh='fsaverage',
    title="Winner-Take-All Somatotopic Map",
    symmetric_cmap=False,
    vmin=0.5,
    vmax=4.5,
    threshold=0.5,
    cmap=cmap_discrete,
    colorbar=True,
    vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"},
)
view.save_as_html(str(FIGURES_DIR / 'winner_take_all_surf.html'))
view